In [2]:
# ─────────────────────────────────────────────
# Cell 1 | Import libraries
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
import joblib

warnings.filterwarnings('ignore')

print("Libraries loaded successfully")


# ─────────────────────────────────────────────
# Cell 2 | Load data & results
# ─────────────────────────────────────────────
lgbm         = joblib.load('../outputs/model_lgbm.pkl')
X_test       = pd.read_csv('../data/X_test.csv')
y_test       = pd.read_csv('../data/y_test.csv').squeeze()
nice_full    = pd.read_csv('../outputs/nice_cf_full.csv')
dice_full    = pd.read_csv('../outputs/dice_cf_full.csv')
nice_summary = pd.read_csv('../outputs/nice_cf_summary.csv')
dice_summary = pd.read_csv('../outputs/dice_cf_summary.csv')

# Variable display name mapping
VAR_LABELS = {
    'sex'                : 'Sex',
    'education'          : 'Education level',
    'marital_status'     : 'Marital status',
    'region'             : 'Region',
    'employment_status'  : 'Employment status',
    'employment_type'    : 'Employment type',
    'h19_pers_income1'   : 'Regular wage income (10k KRW)',
    'h19_pers_income2'   : 'Temporary wage income (10k KRW)',
    'h19_pers_income4'   : 'Side job income (10k KRW)',
    'h19_cin'            : 'Household regular income (10k KRW)',
    'h19_din'            : 'Household disposable income (10k KRW)',
    'debt_financial'     : 'Financial institution loan (10k KRW)',
    'debt_private'       : 'Private loan (10k KRW)',
    'debt_card'          : 'Credit card debt (10k KRW)',
    'debt_lease'         : 'Lease deposit debt (10k KRW)',
    'debt_credit'        : 'Credit purchase debt (10k KRW)',
    'debt_other'         : 'Other debt (10k KRW)',
    'housing_debt_balance': 'Housing debt balance (10k KRW)',
    'housing_tenure'     : 'Housing tenure type',
    'age'                : 'Age',
    'total_debt'         : 'Total debt (10k KRW)',
}

feature_cols = [c for c in nice_full.columns if c not in ['type', 'case_id']]

print(f"NiCE records : {nice_full.shape}")
print(f"DiCE records : {dice_full.shape}")


# ─────────────────────────────────────────────
# Cell 3 | Build NiCE comparison table (long format)
# ─────────────────────────────────────────────
nice_rows = []

for _, summary_row in nice_summary.iterrows():
    case_id   = summary_row['case_id']
    prob_orig = summary_row['prob_original']
    prob_cf   = summary_row['prob_cf']
    changed   = [c.strip() for c in str(summary_row['changed_features']).split(',')]

    sub  = nice_full[nice_full['case_id'] == case_id]
    orig = sub[sub['type'] == 'original'].iloc[0]
    cf   = sub[sub['type'] == 'counterfactual'].iloc[0]

    for feat in changed:
        if feat not in feature_cols:
            continue
        try:
            orig_val = float(orig[feat])
            cf_val   = float(cf[feat])
            change   = cf_val - orig_val
            change_pct = (change / orig_val * 100) if orig_val != 0 else np.nan
        except:
            orig_val   = orig[feat]
            cf_val     = cf[feat]
            change     = np.nan
            change_pct = np.nan

        nice_rows.append({
            'Method'           : 'NiCE',
            'Case ID'          : case_id,
            'Crisis prob (orig)': round(prob_orig, 4),
            'Crisis prob (CF)' : round(prob_cf, 4),
            'Prob reduction'   : round(prob_orig - prob_cf, 4),
            'Feature'          : VAR_LABELS.get(feat, feat),
            'Original value'   : round(orig_val, 2) if isinstance(orig_val, float) else orig_val,
            'CF value'         : round(cf_val, 2) if isinstance(cf_val, float) else cf_val,
            'Change (abs)'     : round(change, 2) if not np.isnan(change) else np.nan,
            'Change (%)'       : round(change_pct, 1) if not np.isnan(change_pct) else np.nan,
        })

nice_df = pd.DataFrame(nice_rows)
print("=== NiCE Comparison Table ===")
print(nice_df.to_string(index=False))


# ─────────────────────────────────────────────
# Cell 4 | Build DiCE comparison table (long format, CF1 only)
# ─────────────────────────────────────────────
dice_rows = []

for case_id in dice_summary['case_id'].unique():
    sub_sum = dice_summary[(dice_summary['case_id'] == case_id) &
                           (dice_summary['cf_id'] == 1)]
    if len(sub_sum) == 0:
        continue
    sub_sum   = sub_sum.iloc[0]
    prob_orig = sub_sum['prob_original']
    prob_cf   = sub_sum['prob_cf']
    changed   = [c.strip() for c in str(sub_sum['changed_features']).split(',')]

    sub_full = dice_full[dice_full['case_id'] == case_id]
    orig     = sub_full[sub_full['type'] == 'original'].iloc[0]
    cf_rows  = sub_full[sub_full['type'] == 'counterfactual']
    if len(cf_rows) == 0:
        continue
    cf = cf_rows.iloc[0]

    for feat in changed:
        if feat not in feature_cols:
            continue
        try:
            orig_val   = float(orig[feat])
            cf_val     = float(cf[feat])
            change     = cf_val - orig_val
            change_pct = (change / orig_val * 100) if orig_val != 0 else np.nan
        except:
            orig_val   = orig[feat]
            cf_val     = cf[feat]
            change     = np.nan
            change_pct = np.nan

        dice_rows.append({
            'Method'            : 'DiCE',
            'Case ID'           : case_id,
            'Crisis prob (orig)': round(prob_orig, 4),
            'Crisis prob (CF)'  : round(prob_cf, 4),
            'Prob reduction'    : round(prob_orig - prob_cf, 4),
            'Feature'           : VAR_LABELS.get(feat, feat),
            'Original value'    : round(orig_val, 2) if isinstance(orig_val, float) else orig_val,
            'CF value'          : round(cf_val, 2) if isinstance(cf_val, float) else cf_val,
            'Change (abs)'      : round(change, 2) if not np.isnan(change) else np.nan,
            'Change (%)'        : round(change_pct, 1) if not np.isnan(change_pct) else np.nan,
        })

dice_df = pd.DataFrame(dice_rows)
print("\n=== DiCE Comparison Table (CF1) ===")
print(dice_df.to_string(index=False))


# ─────────────────────────────────────────────
# Cell 5 | Merge & save all to CSV and Excel
# ─────────────────────────────────────────────
combined_df = pd.concat([nice_df, dice_df], ignore_index=True)

# Save combined CSV
combined_df.to_csv('../outputs/cf_comparison_all.csv',
                   index=False, encoding='utf-8-sig')
print("\nSaved -> outputs/cf_comparison_all.csv")

# Save separate CSVs
nice_df.to_csv('../outputs/cf_comparison_nice.csv',
               index=False, encoding='utf-8-sig')
dice_df.to_csv('../outputs/cf_comparison_dice.csv',
               index=False, encoding='utf-8-sig')
print("Saved -> outputs/cf_comparison_nice.csv")
print("Saved -> outputs/cf_comparison_dice.csv")

# Save Excel with separate sheets
with pd.ExcelWriter('../outputs/cf_comparison_tables.xlsx',
                    engine='openpyxl') as writer:
    nice_df.to_excel(writer, sheet_name='NiCE', index=False)
    dice_df.to_excel(writer, sheet_name='DiCE', index=False)
    combined_df.to_excel(writer, sheet_name='Combined', index=False)

print("Saved -> outputs/cf_comparison_tables.xlsx")


# ─────────────────────────────────────────────
# Cell 6 | NiCE vs DiCE — common cases pivot
#          (same case_id, side by side)
# ─────────────────────────────────────────────
nice_cases = set(nice_df['Case ID'].unique())
dice_cases = set(dice_df['Case ID'].unique())
common     = nice_cases & dice_cases

if len(common) == 0:
    print("No common case IDs between NiCE and DiCE.")
else:
    compare_rows = []
    for case_id in sorted(common):
        n = nice_df[nice_df['Case ID'] == case_id].iloc[0]
        d = dice_df[dice_df['Case ID'] == case_id].iloc[0]
        compare_rows.append({
            'Case ID'                  : case_id,
            'NiCE prob (orig)'         : n['Crisis prob (orig)'],
            'NiCE prob (CF)'           : n['Crisis prob (CF)'],
            'NiCE prob reduction'      : n['Prob reduction'],
            'NiCE changed feature'     : n['Feature'],
            'NiCE original value'      : n['Original value'],
            'NiCE CF value'            : n['CF value'],
            'NiCE change (%)'          : n['Change (%)'],
            'DiCE prob (orig)'         : d['Crisis prob (orig)'],
            'DiCE prob (CF)'           : d['Crisis prob (CF)'],
            'DiCE prob reduction'      : d['Prob reduction'],
            'DiCE changed feature'     : d['Feature'],
            'DiCE original value'      : d['Original value'],
            'DiCE CF value'            : d['CF value'],
            'DiCE change (%)'          : d['Change (%)'],
        })

    compare_df = pd.DataFrame(compare_rows)
    print(f"\n=== Common cases ({len(common)}) — NiCE vs DiCE side-by-side ===")
    print(compare_df.to_string(index=False))

    compare_df.to_csv('../outputs/cf_common_cases.csv',
                      index=False, encoding='utf-8-sig')
    print("\nSaved -> outputs/cf_common_cases.csv")


# ─────────────────────────────────────────────
# Cell 7 | Summary statistics
# ─────────────────────────────────────────────
print("\n=== Summary Statistics ===")
for method, df_m in [('NiCE', nice_df), ('DiCE', dice_df)]:
    if len(df_m) == 0:
        continue
    per_case = df_m.groupby('Case ID').first().reset_index()
    print(f"\n[{method}]")
    print(f"  Cases analyzed      : {df_m['Case ID'].nunique()}")
    print(f"  Avg prob reduction  : {per_case['Prob reduction'].mean():.4f}")
    print(f"  Max prob reduction  : {per_case['Prob reduction'].max():.4f}")
    print(f"  Avg features changed: {df_m.groupby('Case ID').size().mean():.1f}")
    top_feat = df_m['Feature'].value_counts().index[0]
    top_cnt  = df_m['Feature'].value_counts().iloc[0]
    print(f"  Most changed feature: {top_feat} ({top_cnt}x)")

Libraries loaded successfully
NiCE records : (20, 23)
DiCE records : (40, 24)
=== NiCE Comparison Table ===
Method  Case ID  Crisis prob (orig)  Crisis prob (CF)  Prob reduction                              Feature  Original value  CF value  Change (abs)  Change (%)
  NiCE        5              0.9992            0.0461          0.9531 Financial institution loan (10k KRW)         26200.0       0.0      -26200.0      -100.0
  NiCE        5              0.9992            0.0461          0.9531                 Total debt (10k KRW)         26200.0       0.0      -26200.0      -100.0
  NiCE        8              0.0615            0.5640         -0.5025      Temporary wage income (10k KRW)          3310.0    2520.0        -790.0       -23.9
  NiCE       10              0.9879            0.2775          0.7104      Temporary wage income (10k KRW)           240.0    1440.0        1200.0       500.0
  NiCE       11              0.9985            0.1476          0.8509                    Employme